# TP 3 (NLP) - Resoudre la tache par prompting

**Cours NLP et Transformers - CY Tech**

On garde la meme tache (predire le locuteur d'une replique de Tiny Shakespeare),
mais sans entrainer de modele : on decrit la tache a un modele de langage
pre-entraine et on compare deux formulations du prompt.

- zero-shot : la consigne seule ;
- few-shot : la consigne plus quelques exemples.

On utilise un petit modele instruct avec des messages de roles (system, user,
assistant), comme dans la partie 3 du cours. On evalue sur un echantillon du
test et on compare a la reference classique du TP 1. Un GPU accelere l'inference
mais n'est pas indispensable.

In [1]:
!pip install -q transformers

In [2]:
import re
import random
import difflib
import urllib.request

import numpy as np
import torch

from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

NOM_MODELE = "Qwen/Qwen2.5-0.5B-Instruct"

W1023 21:57:27.885000 12932 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## 1. Charger et preparer les donnees

In [3]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
texte = urllib.request.urlopen(url).read().decode("utf-8")


def parser_repliques(txt):
    lignes = []
    for bloc in txt.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        parties = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", parties[0])
        if entete and len(parties) > 1:
            lignes.append((entete.group(1).strip(),
                           " ".join(p.strip() for p in parties[1:]).strip()))
    return lignes


paires = parser_repliques(texte)

from collections import Counter
compte = Counter(locuteur for locuteur, _ in paires)
locuteurs_retenus = [nom for nom, _ in compte.most_common(5)]

paires = [(loc, rep) for loc, rep in paires
          if loc in locuteurs_retenus and len(rep.split()) >= 12]

locuteurs = [loc for loc, _ in paires]
repliques = [rep for _, rep in paires]

X_train, X_test, y_train, y_test = train_test_split(
    repliques, locuteurs, test_size=0.2, stratify=locuteurs, random_state=RANDOM_STATE
)

classes = sorted(set(locuteurs))
print("classes :", classes)
print(len(X_train), "train /", len(X_test), "test")

classes : ['DUKE VINCENTIO', 'GLOUCESTER', 'MENENIUS', 'PETRUCHIO', 'ROMEO']
402 train / 101 test


On tire un echantillon du test pour limiter le temps d'inference.

In [4]:
taille_echantillon = min(80, len(X_test))
indices = random.sample(range(len(X_test)), taille_echantillon)
X_echantillon = [X_test[i] for i in indices]
y_echantillon = [y_test[i] for i in indices]
print(taille_echantillon, "repliques evaluees")

80 repliques evaluees


## 2. Reference classique (TF-IDF + regression logistique)

Meme modele qu'au TP 1, evalue sur le meme echantillon pour pouvoir comparer.

In [5]:
vectoriseur = TfidfVectorizer(lowercase=True, ngram_range=(1, 2),
                              min_df=2, max_df=0.9, sublinear_tf=True)
Xtr = vectoriseur.fit_transform(X_train)
Xech = vectoriseur.transform(X_echantillon)

modele_ml = LogisticRegression(max_iter=1000, class_weight="balanced")
modele_ml.fit(Xtr, y_train)
pred_ml = modele_ml.predict(Xech)

f1_ml = f1_score(y_echantillon, pred_ml, average="macro", labels=classes, zero_division=0)
print("reference ML - f1 macro :", round(f1_ml, 3))

reference ML - f1 macro : 0.691


## 3. Charger le modele de langage

On charge un modele causal instruct via un pipeline de generation de texte. Le
pipeline applique automatiquement le gabarit de conversation du modele.

In [6]:
generateur = pipeline("text-generation", model=NOM_MODELE)
print("modele charge :", NOM_MODELE)

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'Qwen/Qwen2.5-0.5B-Instruct'.


'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct/tree/main/additional_chat_templates


Retrying in 1s [Retry 1/5].


modele charge : Qwen/Qwen2.5-0.5B-Instruct


## 4. Prompting

On ecrit une fonction qui interroge le modele a partir d'une liste de messages
et renvoie uniquement le contenu de la reponse de l'assistant, puis une fonction
qui ramene cette reponse au nom de personnage le plus proche de la liste.

In [7]:
def demander(messages, max_new_tokens=10):
    sortie = generateur(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return sortie[0]["generated_text"][-1]["content"]


def interpreter_reponse(reponse, candidats):
    reponse = reponse.strip().lower()
    for candidat in candidats:
        if candidat.lower() in reponse:
            return candidat
    proches = difflib.get_close_matches(reponse, [c.lower() for c in candidats], n=1)
    if proches:
        for candidat in candidats:
            if candidat.lower() == proches[0]:
                return candidat
    return candidats[0]


consigne = (
    "Tu identifies le personnage d'une piece de Shakespeare qui a prononce une "
    "replique. Personnages possibles : " + ", ".join(classes) + ". "
    "Reponds uniquement par le nom exact du personnage."
)

### 4.1 Zero-shot

La consigne est placee dans le message system ; la replique dans le message
user.

In [8]:
predictions_zs = []
for replique in X_echantillon:
    messages = [
        {"role": "system", "content": consigne},
        {"role": "user", "content": "Replique : " + replique},
    ]
    predictions_zs.append(interpreter_reponse(demander(messages), classes))

f1_zs = f1_score(y_echantillon, predictions_zs, average="macro", labels=classes, zero_division=0)
print("zero-shot - accuracy :", round(accuracy_score(y_echantillon, predictions_zs), 3))
print("zero-shot - f1 macro :", round(f1_zs, 3))

zero-shot - accuracy : 0.263
zero-shot - f1 macro : 0.203


### 4.2 Few-shot

On ajoute un exemple par personnage, tire du train, sous forme de tours de
conversation user puis assistant. Ces exemples consomment du contexte mais
guident le format et la decision.

In [9]:
exemples = []
for classe in classes:
    for replique, locuteur in zip(X_train, y_train):
        if locuteur == classe and len(replique.split()) <= 30:
            exemples.append((replique, locuteur))
            break

messages_exemples = []
for exemple_replique, exemple_locuteur in exemples:
    messages_exemples.append({"role": "user", "content": "Replique : " + exemple_replique})
    messages_exemples.append({"role": "assistant", "content": exemple_locuteur})

predictions_fs = []
for replique in X_echantillon:
    messages = [{"role": "system", "content": consigne}]
    messages += messages_exemples
    messages.append({"role": "user", "content": "Replique : " + replique})
    predictions_fs.append(interpreter_reponse(demander(messages), classes))

f1_fs = f1_score(y_echantillon, predictions_fs, average="macro", labels=classes, zero_division=0)
print("few-shot - accuracy :", round(accuracy_score(y_echantillon, predictions_fs), 3))
print("few-shot - f1 macro :", round(f1_fs, 3))

few-shot - accuracy : 0.287
few-shot - f1 macro : 0.15


## 5. Comparer les approches

In [10]:
print("f1 macro sur le meme echantillon :")
print("  reference ML (TF-IDF + logistique) :", round(f1_ml, 3))
print("  prompting zero-shot                :", round(f1_zs, 3))
print("  prompting few-shot                 :", round(f1_fs, 3))

f1 macro sur le meme echantillon :
  reference ML (TF-IDF + logistique) : 0.691
  prompting zero-shot                : 0.203
  prompting few-shot                 : 0.15


## 6. Observer les parametres de decodage

A chaque pas, le modele produit une distribution sur le prochain token. Les
parametres de decodage decident comment en tirer une suite : choix deterministe
(argmax) ou echantillonnage plus ou moins varie selon la temperature.

In [11]:
messages_demo = [{"role": "user", "content": "Ecris une courte phrase sur la mer."}]

sortie = generateur(messages_demo, max_new_tokens=40, do_sample=False)
print("deterministe :")
print(sortie[0]["generated_text"][-1]["content"])

for temperature in [0.7, 1.3]:
    sortie = generateur(messages_demo, max_new_tokens=40, do_sample=True,
                        temperature=temperature, top_p=0.9)
    print("temperature", temperature, ":")
    print(sortie[0]["generated_text"][-1]["content"])

deterministe :
La mer est un espace immense et inestimable qui offre des richesses naturelles et culturelles inépuisablees. Elle est aussi le lieu de vie de millions d


temperature 0.7 :
"La mer est un oiseau de la nature qui attire les humains avec son histoire et ses vies, et elle reste toujours là, à jamais perdue dans le tumulte du oc


temperature 1.3 :
"La mer est aussi grande que nous ne pensons à elle depuis des milleniums." (From "The Book of Songs," section 36, verse 7)


## 7. Analyse

- Le prompting ne rattrape pas la reference classique : F1 macro 0.691 pour
  TF-IDF + regression logistique, contre 0.203 en zero-shot et 0.150 en
  few-shot. Un petit modele general connait mal le style propre a ces
  personnages.
- Ici le few-shot ne fait pas mieux que le zero-shot : les quelques exemples
  ajoutes allongent le contexte sans apporter de signal fiable, et peuvent meme
  biaiser la reponse vers les personnages montres.
- Les erreurs frequentes sont des confusions entre personnages au registre
  proche, et des reponses hors liste ramenees au candidat le plus proche par la
  fonction d'interpretation.
- Effet de la temperature : en mode deterministe la sortie est stable ; plus la
  temperature monte, plus elle devient variee mais risquee. Pour une tache de
  classification, on prefere un decodage peu aleatoire.
- Un pipeline classique reste preferable quand la tache est etroite, le volume
  important et les contraintes de cout, de latence et de stabilite fortes :
  c'est exactement le cas ici.